# 論文④ revise用 統合再実験ノートブック(G1・多関節インピーダンス摂動/治療優先順位づけ)

**対象論文**: *Ankle-Inclusive Treatment Consistently Restores Gait Stability in Simulated Multi-Joint Spasticity…*
(Frontiers in Robotics and AI、査読対応)

## 旧ノートブック(`g1_spastic_damping_final.ipynb`)からの主な修正点

| # | 問題 | 修正 |
|---|---|---|
| 1 | **方向依存トルクがシミュレーションに届いていなかった**。`raw_env.sim.mj_data.qfrc_applied` はCPU側のコピーで、物理計算はGPU側の `wp_data` に対して `mjwarp.step` で行われる | GPU側の `sim.data.qfrc_applied` に、**物理サブステップ(0.005 s)ごとに**書き込む。§3の検証セルで実際に効いていることを確認してから使う |
| 2 | **速度コマンドが意図どおりでなかった可能性**。方策に渡る観測は `command_manager.compute()` の後に計算されるため、heading環境(30%)では旋回速度が上書きされ、standing環境(10%)ではコマンドがゼロになり、3〜8秒ごとにランダムに再設定される | `heading_command=False`、`rel_heading_envs=0`、`rel_standing_envs=0`、再設定なし、に固定する。§2で旧設定の実態を診断する |
| 3 | 1 seedにつき1回の試行のみ(R2-#6)。同じ条件でも実行のたびに結果が変わる(R2-#10の原因) | **1回の環境構築で並列40環境 = 4速度 × 10試行**。試行ごとに初期状態(位置・高さ・関節角度)と、起動時のドメインランダム化(摩擦・エンコーダのずれ・重心)が異なる |
| 4 | 転倒/非転倒の二値のみ(R1) | 関節可動域、足部の高さ・ピーク、体幹の傾き、前進速度、関節トルク、方策出力を全ステップ記録する |
| 5 | 結果CSVの上書き・セルの実行順の乱れ | 実験ごとに別のCSV(Google Driveに保存、途中から再開可能)。上から順に実行すれば再現できる構成にした |

## 実験と査読コメントの対応

| 節 | 内容 | 対応する指摘 |
|---|---|---|
| §3 | 検証(damping反映、トルク注入、決定性) | R2-#10 の原因説明、方向依存モデルの前提 |
| §4 実験AB | 3関節 damping×10000、治療8条件 × 左右 × 4速度 × 8 seed × 10試行。`treat_all` = 全関節正常 = **無摂動の参照条件** | R1全般、R2-#6, 7, 8, 10, 11 |
| §5 実験D1 | **修正版**方向依存(速度依存)抵抗の用量反応 | R2-#4, #9 |
| §6 実験D2 | 方向依存モデルでの治療優先順位(D1で中程度の転倒率になるkdを使う) | R2-#4、R1(痙縮の構成概念の妥当性) |
| §7 実験C | 単関節(股・膝・足)と内反尖足の用量反応 | R2-#7、本文§3.1–3.2の数値の確認 |
| §8 | 集計・統計・ヒートマップ・運動学の表 | R1-1.5、R2-#11 |
| §9 | 図2の改訂版 | R2-#12 |
| §10 | 結果の書き出し | — |

## 実行の優先順位

**§1 → §2 → §3(必須) → §4 → §5(まずパイロット) → §6 → §7 → §8 → §9 → §10**

どの実験セルも途中で切れたら、**同じセルをそのまま再実行すれば未完了分だけを実行します**。§3.4で1回の環境構築にかかる時間を測り、全体の所要時間を見積もります。

## 1. セットアップ

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
!cp -r "/content/drive/MyDrive/g1_cross_seed_logs/logs" /content/ && echo "復元しました" || echo "復元に失敗しました。パスを確認してください"

In [ ]:
import os
os.environ["MUJOCO_GL"] = "egl"
os.environ["PYOPENGL_PLATFORM"] = ""
os.environ["WANDB_MODE"] = "offline"

%pip install -q "mjlab==1.2.0" "mujoco==3.5.0" "warp-lang==1.12.1" "numpy>=2.0,<2.3" scipy mediapy pandas matplotlib

import torch
assert torch.cuda.is_available(), "GPU ランタイムに切り替えてください"
print("install done:", torch.cuda.get_device_name(0))

In [ ]:
import math, glob, re, time, json, dataclasses
from pathlib import Path
from dataclasses import asdict
import numpy as np
import pandas as pd
import mujoco
import warp as wp
from scipy.signal import find_peaks
from mjlab.envs import ManagerBasedRlEnv
from mjlab.rl import MjlabOnPolicyRunner, RslRlVecEnvWrapper
from mjlab.tasks.registry import load_env_cfg, load_rl_cfg

TASK = "Mjlab-Velocity-Flat-Unitree-G1"
DEVICE = "cuda:0"
ENV_SEED = 100                       # 旧実験と同じ。条件間で同じ乱数系列を使う(対応のある比較になる)
RIGHT_FACING_YAW_RAD = -math.pi / 2

SEEDS = [1, 2, 3, 4, 5, 6, 7, 8]     # 方策の学習seed
SPEEDS = [0.3, 0.5, 0.7, 0.9]        # m/s
N_ROLLOUTS = 10                      # 1 seed・1速度あたりの試行数
STEPS = 250                          # 方策ステップ(50 Hz) = 5.0 s
JOINT_INIT_NOISE = 0.05              # 初期関節角度のランダム化幅 [rad](±約2.9°)
ANALYSIS_START = 25                  # 運動学指標は 0.5 s 以降で計算(起動直後の過渡を除く)
MIN_VALID_STEPS = 50                 # 指標を計算するのに必要な最小ステップ数

OUT_DIR = Path("/content/drive/MyDrive/g1_spasticity_revision")
TS_DIR = OUT_DIR / "timeseries"
OUT_DIR.mkdir(parents=True, exist_ok=True)
TS_DIR.mkdir(parents=True, exist_ok=True)
print("出力先:", OUT_DIR)

In [ ]:
# --- 関節名・自由度番号・物理定数を、プローブ用の環境から確定する ---
_probe_cfg = load_env_cfg(TASK, play=True)
_probe_cfg.scene.num_envs = 1
_probe_env = ManagerBasedRlEnv(cfg=_probe_cfg, device=DEVICE, render_mode=None)
_robot = _probe_env.scene["robot"]
JOINT_NAMES = list(_robot.joint_names)
BODY_NAMES = list(_robot.body_names)
assert len(JOINT_NAMES) == 29
_mjm = _probe_env.sim.mj_model
DOF_ADR = {jn: int(_mjm.jnt_dofadr[mujoco.mj_name2id(_mjm, mujoco.mjtObj.mjOBJ_JOINT, f"robot/{jn}")])
           for jn in JOINT_NAMES}
PHYSICS_DT = float(_probe_env.physics_dt)
STEP_DT = float(_probe_env.step_dt)
DECIMATION = int(round(STEP_DT / PHYSICS_DT))
_probe_env.close(); del _probe_env, _robot; torch.cuda.empty_cache()

# 関節ごとのアクチュエータ上限トルク(effort_limit)
EFFORT_LIMIT = {}
for _act in _probe_cfg.scene.entities["robot"].articulation.actuators:
    for jn in JOINT_NAMES:
        if any(re.fullmatch(p, jn) for p in _act.target_names_expr):
            EFFORT_LIMIT[jn] = float(_act.effort_limit) if _act.effort_limit is not None else float("inf")

# 解析対象の関節(脚の矢状面3関節 × 左右 + 足関節roll)
LEG_JOINTS = [f"{s}_{j}" for s in ("left", "right")
              for j in ("hip_pitch_joint", "knee_joint", "ankle_pitch_joint")] + \
             ["left_ankle_roll_joint", "right_ankle_roll_joint"]
LEG_JIDX = [JOINT_NAMES.index(j) for j in LEG_JOINTS]
LEG_DOF = [DOF_ADR[j] for j in LEG_JOINTS]
FOOT_BODY_IDX = [BODY_NAMES.index("left_ankle_roll_link"), BODY_NAMES.index("right_ankle_roll_link")]

print(f"physics_dt={PHYSICS_DT} s, step_dt={STEP_DT} s, decimation={DECIMATION}")
print(f"{STEPS} 方策ステップ = {STEPS*STEP_DT:.2f} s")
print("effort_limit [N·m]:", {j: EFFORT_LIMIT[j] for j in LEG_JOINTS})

In [ ]:
def find_checkpoint(seed):
    run_dirs = sorted(glob.glob(f"logs/rsl_rl/g1_velocity/*_seed{seed}"))
    if not run_dirs:
        raise FileNotFoundError(f"seed{seed}: runディレクトリが見つかりません。")
    return max(Path(run_dirs[-1]).glob("model_*.pt"),
               key=lambda p: int(re.search(r"model_(\d+)", p.name).group(1)))

CKPT_INFO = {s: str(find_checkpoint(s)) for s in SEEDS}
for s, p in CKPT_INFO.items():
    print(f"seed{s}: {p}")

## 2. 診断:旧設定で、方策が実際に見ていた速度コマンド

旧ノートブックと**まったく同じ設定**(num_envs=1、ENV_SEED=100、コマンド設定は既定のまま)で環境を作ります。
各ステップの後に、方策の観測に入ったコマンド(`vel_command_b`)を記録します。

- `is_heading_env=True` の場合:旋回速度が、ランダムな目標方位へ向かう制御で上書きされていた
- `is_standing_env=True` の場合:コマンドが常にゼロになり、方策は「立ち止まれ」という指示を見ていた

**この結果は、論文の記述(コマンド速度)が正しかったかどうかの確認に必要です。結果を共有してください。**

In [ ]:
def diagnose_original_command(seed, speed=0.5, steps=STEPS):
    play_cfg = load_env_cfg(TASK, play=True)
    play_cfg.scene.num_envs = 1
    play_cfg.seed = ENV_SEED
    play_cfg.events["reset_base"].params["pose_range"]["yaw"] = (RIGHT_FACING_YAW_RAD, RIGHT_FACING_YAW_RAD)
    agent_cfg = load_rl_cfg(TASK)
    raw_env = ManagerBasedRlEnv(cfg=play_cfg, device=DEVICE, render_mode=None)
    env = RslRlVecEnvWrapper(raw_env, clip_actions=agent_cfg.clip_actions)
    runner = MjlabOnPolicyRunner(env, asdict(agent_cfg), device=DEVICE)
    runner.load(CKPT_INFO[seed], load_cfg={"actor": True}, strict=True, map_location=DEVICE)
    policy = runner.get_inference_policy(device=DEVICE)
    twist = raw_env.command_manager.get_term("twist")
    robot = raw_env.scene["robot"]
    info = {"seed": seed,
            "is_heading_env": bool(twist.is_heading_env[0]),
            "is_standing_env": bool(twist.is_standing_env[0]),
            "heading_target_deg": float(torch.rad2deg(twist.heading_target[0])),
            "time_left_s": float(twist.time_left[0])}
    forward_cmd = torch.tensor([[speed, 0.0, 0.0]], device=DEVICE)
    obs = env.get_observations()
    rows = []
    with torch.no_grad():
        for t in range(steps):
            twist.vel_command_b[:] = forward_cmd           # 旧コードと同じ上書き
            obs, _, dones, _ = env.step(policy(obs))
            c = twist.vel_command_b[0].cpu().numpy()        # ← 次の行動を決める観測に入ったコマンド
            rows.append({"step": t, "cmd_vx": c[0], "cmd_vy": c[1], "cmd_wz": c[2],
                         "base_vx": float(robot.data.root_link_lin_vel_b[0, 0]),
                         "base_wz": float(robot.data.root_link_ang_vel_b[0, 2]),
                         "done": bool(dones[0])})
            if bool(dones[0]):
                break
    env.close(); del runner, policy, env, raw_env; torch.cuda.empty_cache()
    df = pd.DataFrame(rows)
    ok = (np.isclose(df.cmd_vx, speed, atol=1e-4) & np.isclose(df.cmd_vy, 0, atol=1e-4)
          & np.isclose(df.cmd_wz, 0, atol=1e-4))
    info["frac_steps_command_as_intended"] = float(ok.mean())
    info["mean_cmd_wz_seen"] = float(df.cmd_wz.mean())
    info["mean_base_vx"] = float(df.base_vx.mean())
    info["mean_base_wz_deg_s"] = float(np.rad2deg(df.base_wz.mean()))
    return info, df

diag = []
for s in [1, 8]:
    info, _df = diagnose_original_command(s)
    diag.append(info)
diag_df = pd.DataFrame(diag)
diag_df.to_csv(OUT_DIR / "diag_original_command.csv", index=False)
print(diag_df.T.to_string())

## 3. 共通関数と検証

### 3.1 環境の作成(コマンドの固定、初期状態のランダム化、dampingの変更)

In [ ]:
def build_actuators_with_modified_joint(base_actuators, joint_name, damping_scale=1.0, stiffness_scale=1.0):
    # 旧ノートブックと同じ(検証済み)。joint_name だけをグループから切り出してゲインを変更する
    new_actuators = []
    for act in base_actuators:
        matched = [jn for jn in JOINT_NAMES if any(re.fullmatch(p, jn) for p in act.target_names_expr)]
        if joint_name in matched:
            remaining = [jn for jn in matched if jn != joint_name]
            if remaining:
                new_actuators.append(dataclasses.replace(act, target_names_expr=tuple(remaining)))
            new_actuators.append(dataclasses.replace(
                act, target_names_expr=(joint_name,),
                damping=act.damping * damping_scale, stiffness=act.stiffness * stiffness_scale))
        else:
            new_actuators.append(act)
    return tuple(new_actuators)


def make_play_cfg(num_envs, joint_damping=None, init_noise=JOINT_INIT_NOISE, env_seed=ENV_SEED):
    # joint_damping: {関節名: damping倍率}
    cfg = load_env_cfg(TASK, play=True)
    cfg.scene.num_envs = num_envs
    cfg.seed = env_seed
    cfg.events["reset_base"].params["pose_range"]["yaw"] = (RIGHT_FACING_YAW_RAD, RIGHT_FACING_YAW_RAD)
    cfg.events["reset_robot_joints"].params["position_range"] = (-init_noise, init_noise)
    # --- 修正点2: 速度コマンドを固定する ---
    tw = cfg.commands["twist"]
    tw.heading_command = False
    tw.ranges.heading = None          # heading_command=False のとき、mjlabは None を要求する
    tw.rel_heading_envs = 0.0
    tw.rel_standing_envs = 0.0
    tw.resampling_time_range = (1.0e9, 1.0e9)
    tw.init_velocity_prob = 0.0
    if joint_damping:
        acts = cfg.scene.entities["robot"].articulation.actuators
        for jn, sc in joint_damping.items():
            if sc != 1.0:
                acts = build_actuators_with_modified_joint(acts, jn, damping_scale=sc)
        cfg.scene.entities["robot"].articulation.actuators = acts
    return cfg


def build_env_and_policy(seed, cfg, render_mode=None):
    agent_cfg = load_rl_cfg(TASK)
    raw_env = ManagerBasedRlEnv(cfg=cfg, device=DEVICE, render_mode=render_mode)
    env = RslRlVecEnvWrapper(raw_env, clip_actions=agent_cfg.clip_actions)
    runner = MjlabOnPolicyRunner(env, asdict(agent_cfg), device=DEVICE)
    runner.load(CKPT_INFO[seed], load_cfg={"actor": True}, strict=True, map_location=DEVICE)
    policy = runner.get_inference_policy(device=DEVICE)
    return runner, policy, env, raw_env


def actuator_kd_for_joint(raw_env, joint_name):
    # MuJoCoモデル上で、その関節を駆動するアクチュエータの kp, kd を返す(damping変更の確認用)
    m = raw_env.sim.mj_model
    jid = mujoco.mj_name2id(m, mujoco.mjtObj.mjOBJ_JOINT, f"robot/{joint_name}")
    aids = [a for a in range(m.nu) if int(m.actuator_trnid[a, 0]) == jid]
    return [(float(m.actuator_gainprm[a, 0]), float(-m.actuator_biasprm[a, 2])) for a in aids]

print("定義しました。")

### 3.2 方向依存(速度依存)抵抗の注入 — **修正版**

`sim.step` を差し替え、**物理サブステップごと**に次を行います。

1. GPU側の `sim.data.qvel` から、対象関節の角速度を読む
2. 痙縮筋が伸張される方向に動いているときだけ、`τ = −kd·ω` を計算する
3. 関節ごとの上限(既定はアクチュエータの `effort_limit`)でクリップする
4. GPU側の `sim.data.qfrc_applied` に書き込む

抵抗の方向は旧ノートブック§10.1bの目視確認に従います。股関節屈筋は伸展方向(ω>0)、膝伸筋は屈曲方向(ω>0)、足底屈筋は背屈方向(ω<0)で抵抗します。

上限なしにしない理由:`qfrc_applied` は陽的に積分されるため、kdが大きいと数値的に発散します(MuJoCoのアクチュエータ減衰とは違い、陰的に扱われません)。アクチュエータの上限トルクは、物理的に意味のある上限として使えます。

In [ ]:
DIRECTIONAL_CONFIG = {
    "hip":   {"joint": "hip_pitch_joint",   "resist_sign": +1},
    "knee":  {"joint": "knee_joint",        "resist_sign": +1},
    "ankle": {"joint": "ankle_pitch_joint", "resist_sign": -1},
}

def _tensor(x):
    return x._tensor if hasattr(x, "_tensor") else x


class TorqueInjector:
    # mode="directional": τ = -kd·ω(伸張方向のときのみ)、mode="constant": τ = const(検証用)
    def __init__(self, raw_env, joint_names, resist_signs=None, kd=0.0, max_torque=None,
                 mode="directional", constant_torque=0.0):
        self.sim = raw_env.sim
        self.n = raw_env.num_envs
        self.joint_names = list(joint_names)
        self.dofs = torch.tensor([DOF_ADR[j] for j in self.joint_names], device=DEVICE, dtype=torch.long)
        k = len(self.joint_names)
        self.signs = torch.tensor(resist_signs if resist_signs is not None else [1] * k,
                                  device=DEVICE, dtype=torch.float32)
        caps = max_torque if max_torque is not None else [EFFORT_LIMIT[j] for j in self.joint_names]
        if np.isscalar(caps):
            caps = [caps] * k
        self.cap = torch.tensor(caps, device=DEVICE, dtype=torch.float32)
        self.kd, self.mode, self.const = float(kd), mode, float(constant_torque)
        self.last_tau = torch.zeros(self.n, k, device=DEVICE)
        self.stream = torch.cuda.ExternalStream(wp.get_stream(DEVICE).cuda_stream)
        self._orig_step = self.sim.step
        self.sim.step = self._step                     # インスタンス属性で差し替え

    def _step(self):
        with torch.cuda.stream(self.stream):
            qvel = _tensor(self.sim.data.qvel)[:, self.dofs]
            if self.mode == "directional":
                tau = torch.where(qvel * self.signs > 0, -self.kd * qvel, torch.zeros_like(qvel))
            else:
                tau = torch.full_like(qvel, self.const)
            tau = torch.maximum(torch.minimum(tau, self.cap), -self.cap)
            _tensor(self.sim.data.qfrc_applied)[:, self.dofs] = tau
            self.last_tau = tau
        self._orig_step()

    def remove(self):
        with torch.cuda.stream(self.stream):
            _tensor(self.sim.data.qfrc_applied)[:, self.dofs] = 0.0
        self.sim.step = self._orig_step


def make_directional_injector(raw_env, side, joints, kd, cap="effort", treated=()):
    active = [j for j in joints if j not in treated]
    if not active:
        return None
    names = [f"{side}_{DIRECTIONAL_CONFIG[j]['joint']}" for j in active]
    signs = [DIRECTIONAL_CONFIG[j]["resist_sign"] for j in active]
    max_t = None if cap == "effort" else cap
    return TorqueInjector(raw_env, names, signs, kd=kd, max_torque=max_t, mode="directional")

print("定義しました。")

### 3.3 バッチ実行と指標の計算

1回の環境構築で `len(speeds_per_env)` 個の環境を並列に動かします。各環境について、**最初に転倒した時点で記録を打ち切ります**(その後は自動リセットされるので使いません)。転倒の判定は、胴体の傾きが70°を超えたとき(mjlabの `bad_orientation`)です。

In [ ]:
TS_FIELDS = ["qpos_deg", "tau_act", "tau_dir", "foot_z", "proj_grav", "base_vx", "action"]

def run_batch(seed, speeds_per_env, joint_damping=None, directional=None, steps=STEPS,
              init_noise=JOINT_INIT_NOISE, env_seed=ENV_SEED, check_kd_joint=None):
    # directional: None または dict(side=, joints=, kd=, cap=, treated=)
    n = len(speeds_per_env)
    cfg = make_play_cfg(n, joint_damping, init_noise, env_seed)
    runner, policy, env, raw_env = build_env_and_policy(seed, cfg)
    if check_kd_joint is not None:
        print(f"  [確認] {check_kd_joint} の (kp, kd):", actuator_kd_for_joint(raw_env, check_kd_joint))
    robot = raw_env.scene["robot"]
    twist = raw_env.command_manager.get_term("twist")
    cmd = torch.zeros(n, 3, device=DEVICE)
    cmd[:, 0] = torch.tensor(speeds_per_env, device=DEVICE, dtype=torch.float32)
    twist.vel_command_b[:] = cmd
    inj = make_directional_injector(raw_env, **directional) if directional else None
    dir_cols = []
    if inj is not None:
        dir_cols = [LEG_JOINTS.index(j) for j in inj.joint_names]

    jidx = torch.tensor(LEG_JIDX, device=DEVICE)
    dofs = torch.tensor(LEG_DOF, device=DEVICE)
    feet = torch.tensor(FOOT_BODY_IDX, device=DEVICE)
    alive = torch.ones(n, dtype=torch.bool, device=DEVICE)
    fall_step = torch.full((n,), -1, dtype=torch.long, device=DEVICE)
    buf = {k: [] for k in TS_FIELDS}
    valid = []

    obs = env.get_observations()               # コマンド固定後に観測を計算し直す
    with torch.no_grad():
        for t in range(steps):
            twist.vel_command_b[:] = cmd
            action = policy(obs)
            obs, _, dones, _ = env.step(action)
            done = dones.bool()
            fall_step[alive & done] = t
            ok = alive & ~done                 # このステップの状態が有効(転倒前)な環境
            valid.append(ok.cpu().numpy())
            buf["qpos_deg"].append(torch.rad2deg(robot.data.joint_pos[:, jidx]).cpu().numpy())
            buf["tau_act"].append(_tensor(raw_env.sim.data.qfrc_actuator)[:, dofs].cpu().numpy())
            td = np.zeros((n, len(LEG_JOINTS)), dtype=np.float32)
            if inj is not None:
                td[:, dir_cols] = inj.last_tau.cpu().numpy()
            buf["tau_dir"].append(td)
            buf["foot_z"].append(robot.data.body_link_pos_w[:, feet, 2].cpu().numpy())
            buf["proj_grav"].append(robot.data.projected_gravity_b.cpu().numpy())
            buf["base_vx"].append(robot.data.root_link_lin_vel_b[:, 0:1].cpu().numpy())
            buf["action"].append(action[:, jidx].cpu().numpy())
            alive &= ~done
            if not alive.any():
                break

    if inj is not None:
        inj.remove()
    env.close(); del runner, policy, env, raw_env; torch.cuda.empty_cache()
    ts = {k: np.stack(v, axis=1).astype(np.float32) for k, v in buf.items()}   # (n, T, ...)
    ts["valid"] = np.stack(valid, axis=1)
    ts["fall_step"] = fall_step.cpu().numpy()
    ts["speed"] = np.asarray(speeds_per_env, dtype=np.float32)
    return ts


def _peaks(z):
    if len(z) < 12:
        return np.array([], dtype=int)
    p, _ = find_peaks(z, distance=10, prominence=0.005)
    return p


def summarize_env(ts, i, affected_side=None, small_peak_thresh=0.05):
    v = ts["valid"][i]
    n_valid = int(v.sum())
    fs = int(ts["fall_step"][i])
    out = {"fell": fs >= 0, "fall_step": fs,
           "time_to_fall_s": (fs + 1) * STEP_DT if fs >= 0 else np.nan,
           "n_valid_steps": n_valid}
    w = np.where(v)[0]
    w = w[w >= ANALYSIS_START]
    if len(w) < MIN_VALID_STEPS:
        return out
    q = ts["qpos_deg"][i, w]
    ta = ts["tau_act"][i, w]
    td = ts["tau_dir"][i, w]
    act = ts["action"][i, w]
    for c, jn in enumerate(LEG_JOINTS):
        key = jn.replace("_joint", "")
        out[f"rom_{key}"] = float(q[:, c].max() - q[:, c].min())
        out[f"tauabs_{key}"] = float(np.abs(ta[:, c]).mean())
        out[f"taupk_{key}"] = float(np.abs(ta[:, c]).max())
        lim = EFFORT_LIMIT[jn]
        out[f"sat_{key}"] = float((np.abs(ta[:, c]) >= 0.98 * lim).mean()) if np.isfinite(lim) else 0.0
        out[f"taudir_{key}"] = float(np.abs(td[:, c]).mean())
        out[f"actstd_{key}"] = float(act[:, c].std())
        out[f"actmean_{key}"] = float(act[:, c].mean())
    for fi, side in enumerate(("left", "right")):
        z = ts["foot_z"][i, w, fi]
        p = _peaks(z)
        ph = z[p]
        out[f"foot_{side}_peak_mean"] = float(ph.mean()) if len(ph) else np.nan
        out[f"foot_{side}_n_main_peaks"] = int((ph >= small_peak_thresh).sum())
        out[f"foot_{side}_n_small_peaks"] = int((ph < small_peak_thresh).sum())
        out[f"foot_{side}_main_peak_mean"] = float(ph[ph >= small_peak_thresh].mean()) if (ph >= small_peak_thresh).any() else np.nan
    g = ts["proj_grav"][i, w]
    roll = np.degrees(np.arctan2(g[:, 1], -g[:, 2]))
    pitch = np.degrees(np.arctan2(-g[:, 0], -g[:, 2]))
    out["trunk_roll_std_deg"] = float(roll.std())
    out["trunk_pitch_std_deg"] = float(pitch.std())
    out["trunk_tilt_max_deg"] = float(np.degrees(np.arccos(np.clip(-g[:, 2], -1, 1))).max())
    vx = ts["base_vx"][i, w, 0]
    out["base_vx_mean"] = float(vx.mean())
    out["vx_tracking_err"] = float(abs(vx.mean() - ts["speed"][i]))
    return out


def summarize_batch(ts, meta, affected_side=None):
    rows = []
    for i in range(len(ts["speed"])):
        r = dict(meta)
        r["speed"] = float(ts["speed"][i])
        r["rollout"] = int(i % N_ROLLOUTS)
        r.update(summarize_env(ts, i, affected_side))
        rows.append(r)
    return pd.DataFrame(rows)


def speeds_layout(speeds=SPEEDS, n_rollouts=N_ROLLOUTS):
    # 環境 i の速度 = speeds[i // n_rollouts]
    return [s for s in speeds for _ in range(n_rollouts)]


def run_grid(exp_name, jobs, save_ts=True):
    # jobs: dictのリスト。各dictは key(文字列)、meta(dict)、seed、run_batch用のkwargsを持つ
    csv = OUT_DIR / f"{exp_name}.csv"
    done_keys = set(pd.read_csv(csv)["job_key"].unique()) if csv.exists() else set()
    todo = [j for j in jobs if j["key"] not in done_keys]
    print(f"[{exp_name}] 全{len(jobs)}ジョブ中、完了済み{len(jobs)-len(todo)}、残り{len(todo)}")
    t0 = time.time()
    for k, job in enumerate(todo):
        t1 = time.time()
        ts = run_batch(job["seed"], **job["kwargs"])
        df = summarize_batch(ts, {**job["meta"], "seed": job["seed"], "job_key": job["key"]})
        df.to_csv(csv, mode="a", header=not csv.exists(), index=False)
        if save_ts:
            np.savez_compressed(TS_DIR / f"{exp_name}__{job['key']}.npz", **ts)
        el = time.time() - t1
        rem = (time.time() - t0) / (k + 1) * (len(todo) - k - 1)
        print(f"  [{k+1}/{len(todo)}] {job['key']}: 転倒 {int(df['fell'].sum())}/{len(df)}"
              f"  ({el:.0f} s、残り約 {rem/60:.0f} 分)")
    return pd.read_csv(csv) if csv.exists() else pd.DataFrame()

print("定義しました。")

### 3.4 検証(**必ず実行して、結果を確認してください**)

1. **damping の反映**:3関節 damping×10000 の環境で、MuJoCoモデル上の kd が10000倍になっているか
2. **トルク注入が効いているか**:速度0で立たせ、右膝に一定トルク(+20 N·m)をかけた場合とかけない場合で、膝の角度が明確に変わるか。**ここで差が出なければ、方向依存の実験(§5・§6)は実行しないでください**
3. **決定性**:同じ条件を2回実行し、40環境の転倒結果がどれだけ一致するか(R2-#10の説明に使います)
4. **所要時間**:1ジョブ(40環境 × 250ステップ)にかかる時間

In [ ]:
# 検証1: damping の反映
t_start = time.time()
ts_chk = run_batch(1, [0.5] * 2, steps=5,
                   joint_damping={"right_hip_pitch_joint": 1e4, "right_knee_joint": 1e4, "right_ankle_pitch_joint": 1e4},
                   check_kd_joint="right_ankle_pitch_joint")
print("(元の値はkd≈1.8144。10000倍で≈18144になっていればOK)")

In [ ]:
# 検証2: トルク注入が実際に効いているか(速度0で立位)
def injection_test(seed=1, torque=20.0, joint="right_knee_joint", steps=100):
    res = {}
    for label, tq in [("no_torque", 0.0), ("with_torque", torque)]:
        cfg = make_play_cfg(1, None, init_noise=0.0)
        runner, policy, env, raw_env = build_env_and_policy(seed, cfg)
        twist = raw_env.command_manager.get_term("twist")
        twist.vel_command_b[:] = 0.0
        inj = TorqueInjector(raw_env, [joint], mode="constant", constant_torque=tq,
                             max_torque=[1e6])
        robot = raw_env.scene["robot"]
        ji = JOINT_NAMES.index(joint)
        obs = env.get_observations()
        angles = []
        with torch.no_grad():
            for t in range(steps):
                twist.vel_command_b[:] = 0.0
                obs, _, dones, _ = env.step(policy(obs))
                angles.append(float(torch.rad2deg(robot.data.joint_pos[0, ji])))
        applied = float(_tensor(raw_env.sim.data.qfrc_applied)[0, DOF_ADR[joint]])
        inj.remove()
        env.close(); del runner, policy, env, raw_env; torch.cuda.empty_cache()
        res[label] = {"mean_angle_last50_deg": float(np.mean(angles[-50:])), "qfrc_applied_read_back": applied}
    diff = res["with_torque"]["mean_angle_last50_deg"] - res["no_torque"]["mean_angle_last50_deg"]
    print(json.dumps(res, indent=1))
    print(f"膝角度の差 = {diff:+.2f}°")
    if abs(diff) < 0.5:
        print("⚠️ トルク注入が効いていない可能性があります。§5・§6を実行する前に連絡してください。")
    else:
        print("✅ トルク注入はシミュレーションに反映されています。")
    return res, diff

inj_res, inj_diff = injection_test()
pd.DataFrame([{"condition": k, "knee_angle_mean_last50_deg": v["mean_angle_last50_deg"],
               "qfrc_applied_read_back_Nm": v["qfrc_applied_read_back"]} for k, v in inj_res.items()]
             ).to_csv(OUT_DIR / "verification_torque_injection.csv", index=False)

In [ ]:
# 検証3・4: 決定性と所要時間(seed1、3関節 damping×10000、右側、治療なし、40環境 × 2回)
_jd = {"right_hip_pitch_joint": 1e4, "right_knee_joint": 1e4, "right_ankle_pitch_joint": 1e4}
_t = time.time(); tsA = run_batch(1, speeds_layout(), joint_damping=_jd); t_job = time.time() - _t
tsB = run_batch(1, speeds_layout(), joint_damping=_jd)
fa, fb = tsA["fall_step"] >= 0, tsB["fall_step"] >= 0
agree = float((fa == fb).mean())
print(f"1ジョブ(40環境×{STEPS}ステップ)の所要時間: {t_job:.0f} s")
print(f"同一条件2回の転倒結果の一致率: {agree*100:.1f}%  (1回目の転倒 {fa.sum()}/40、2回目 {fb.sum()}/40)")
for sp in SPEEDS:
    m = tsA["speed"] == sp
    print(f"  {sp} m/s: 1回目 {fa[m].mean():.2f}、2回目 {fb[m].mean():.2f}")
pd.DataFrame([{"t_job_s": t_job, "agreement": agree, "falls_run1": int(fa.sum()), "falls_run2": int(fb.sum())}]
             ).to_csv(OUT_DIR / "verification_determinism.csv", index=False)

n_jobs = {"AB": 8 * 2 * len(SEEDS), "D1": 2 * 6 * 2 * len(SEEDS), "D2": 8 * 2 * len(SEEDS), "C": 4 * 7 * 2 * len(SEEDS)}
for k, v in n_jobs.items():
    print(f"実験{k}: {v}ジョブ ≈ {v * t_job / 3600:.1f} 時間")

## 4. 実験AB:3関節 damping×10000 と、治療8条件

- 3関節(股関節pitch・膝・足関節pitch)の damping ×10000 を基準状態とし、治療した関節は正常値に戻す
- `treat_all` は全関節が正常 = **無摂動の参照条件**(R2-#11)。左右で同じ設定なので、2回の独立した実行にもなる
- 左右 × 8 seed × 8条件 = 128ジョブ。1ジョブで4速度 × 10試行

In [ ]:
DAMPING_3J = 10000.0
TREATMENTS = {
    "no_treatment": (), "treat_hip": ("hip",), "treat_knee": ("knee",), "treat_ankle": ("ankle",),
    "treat_hip_knee": ("hip", "knee"), "treat_hip_ankle": ("hip", "ankle"),
    "treat_knee_ankle": ("knee", "ankle"), "treat_all": ("hip", "knee", "ankle"),
}
J3 = {"hip": "hip_pitch_joint", "knee": "knee_joint", "ankle": "ankle_pitch_joint"}

def damping_map(side, treated, scale=DAMPING_3J):
    return {f"{side}_{J3[j]}": scale for j in J3 if j not in treated}

jobs_AB = []
for side in ["right", "left"]:
    for cond, treated in TREATMENTS.items():
        for seed in SEEDS:
            jobs_AB.append({"key": f"{side}_{cond}_s{seed}", "seed": seed,
                            "meta": {"side": side, "condition": cond, "model": "symmetric_damping",
                                     "damping_scale": DAMPING_3J},
                            "kwargs": {"speeds_per_env": speeds_layout(),
                                       "joint_damping": damping_map(side, treated)}})
df_AB = run_grid("expAB_symmetric_priority", jobs_AB)

## 5. 実験D1:方向依存(速度依存)抵抗の用量反応 — 修正版

- 関節の組:`ankle`(足関節のみ)、`three`(股・膝・足関節)
- kd:10, 30, 100, 300, 1000, 3000 [N·m·s/rad]。上限はアクチュエータの effort_limit
- **まず `D1_PILOT=True` で seed 1・8 だけ実行**し、転倒が起き始めるkdを確認してから、`False` にして全seedを実行してください

In [ ]:
D1_PILOT = True
D1_KDS = [10, 30, 100, 300, 1000, 3000]
D1_SETS = {"ankle": ("ankle",), "three": ("hip", "knee", "ankle")}
d1_seeds = [1, 8] if D1_PILOT else SEEDS

jobs_D1 = []
for side in ["right", "left"]:
    for set_name, joints in D1_SETS.items():
        for kd in D1_KDS:
            for seed in d1_seeds:
                jobs_D1.append({"key": f"{side}_{set_name}_kd{kd}_s{seed}", "seed": seed,
                                "meta": {"side": side, "joint_set": set_name, "kd": kd, "cap": "effort",
                                         "model": "directional"},
                                "kwargs": {"speeds_per_env": speeds_layout(),
                                           "directional": {"side": side, "joints": joints, "kd": kd, "cap": "effort"}}})
df_D1 = run_grid("expD1_directional_dose", jobs_D1)
print(df_D1.groupby(["side", "joint_set", "kd", "speed"])["fell"].mean().unstack().round(3).to_string())

## 6. 実験D2:方向依存モデルでの治療優先順位

§5の結果から、3関節(`three`)で**治療なしの転倒率が0.5 m/sで30〜60%程度**になるkdを `D2_KD` に設定してください(迷ったら結果を共有してください)。

In [ ]:
D2_KD = None   # ← §5の結果を見て設定する(例: 300)
assert D2_KD is not None, "D2_KD を設定してください"

jobs_D2 = []
for side in ["right", "left"]:
    for cond, treated in TREATMENTS.items():
        for seed in SEEDS:
            jobs_D2.append({"key": f"{side}_{cond}_s{seed}", "seed": seed,
                            "meta": {"side": side, "condition": cond, "model": "directional", "kd": D2_KD},
                            "kwargs": {"speeds_per_env": speeds_layout(),
                                       "directional": {"side": side, "joints": ("hip", "knee", "ankle"),
                                                       "kd": D2_KD, "cap": "effort", "treated": treated}}})
df_D2 = run_grid("expD2_directional_priority", jobs_D2)

## 7. 実験C:単関節と内反尖足の用量反応(対称damping)

R2-#7への対応と、本文§3.1–3.2の数値(内反尖足の「5000倍で75%」など)の確認です。倍率1(無摂動)は実験ABの `treat_all` を使います。

In [ ]:
C_SCALES = [10, 100, 500, 1000, 2000, 5000, 10000]
C_TARGETS = {"ankle_pitch": ["ankle_pitch_joint"], "hip_pitch": ["hip_pitch_joint"],
             "knee": ["knee_joint"], "equinovarus": ["ankle_pitch_joint", "ankle_roll_joint"]}
jobs_C = []
for side in ["right", "left"]:
    for tgt, jlist in C_TARGETS.items():
        for sc in C_SCALES:
            for seed in SEEDS:
                jobs_C.append({"key": f"{side}_{tgt}_x{sc}_s{seed}", "seed": seed,
                               "meta": {"side": side, "target": tgt, "damping_scale": sc, "model": "symmetric_damping"},
                               "kwargs": {"speeds_per_env": speeds_layout(),
                                          "joint_damping": {f"{side}_{j}": float(sc) for j in jlist}}})
df_C = run_grid("expC_single_joint_dose", jobs_C, save_ts=False)

## 8. 集計・統計・図

- **seedごとの転倒確率** = そのseedの10試行に占める転倒の割合。表は8 seedの平均 ± SDと95% CI
- 治療の効果:seedごとの転倒確率で、治療なしとの対応のあるWilcoxon符号順位検定
- 回復(restoration):参照条件(`treat_all` = 無摂動)との差。差の95% CIの上限が +5ポイント以下なら「回復」と判定

In [ ]:
import matplotlib.pyplot as plt
from scipy import stats

COND_ORDER = list(TREATMENTS.keys())

def per_seed(df, keys):
    return df.groupby(keys + ["seed"])["fell"].mean().rename("p_fall").reset_index()

def ci95(x):
    x = np.asarray(x, float); n = len(x)
    if n < 2:
        return (np.nan, np.nan)
    h = stats.t.ppf(0.975, n - 1) * x.std(ddof=1) / np.sqrt(n)
    return (x.mean() - h, x.mean() + h)

def fall_table(df, row="condition"):
    ps = per_seed(df, ["side", row, "speed"])
    out = []
    for (side, r, sp), g in ps.groupby(["side", row, "speed"]):
        lo, hi = ci95(g.p_fall)
        out.append({"side": side, row: r, "speed": sp, "mean": g.p_fall.mean(), "sd": g.p_fall.std(ddof=1),
                    "ci_lo": lo, "ci_hi": hi, "n_seeds": len(g)})
    return pd.DataFrame(out)

def treatment_tests(df, ref="no_treatment", restore_ref="treat_all", margin=0.05):
    ps = per_seed(df, ["side", "condition", "speed"])
    piv = ps.pivot_table(index=["side", "speed", "seed"], columns="condition", values="p_fall").reset_index()
    rows = []
    for (side, sp), g in piv.groupby(["side", "speed"]):
        for c in COND_ORDER:
            if c not in g or c == ref:
                continue
            d = g[c] - g[ref]
            try:
                p = stats.wilcoxon(g[c], g[ref], zero_method="zsplit").pvalue if (d != 0).any() else 1.0
            except ValueError:
                p = np.nan
            dr = g[c] - g[restore_ref]
            lo, hi = ci95(dr)
            rows.append({"side": side, "speed": sp, "condition": c,
                         "p_fall": g[c].mean(), "delta_vs_untreated": d.mean(),
                         "n_improved": int((d < 0).sum()), "n_worsened": int((d > 0).sum()),
                         "wilcoxon_p": p, "delta_vs_reference": dr.mean(), "delta_ref_ci_hi": hi,
                         "restored": bool(hi <= margin) if np.isfinite(hi) else bool(dr.mean() <= margin)})
    return pd.DataFrame(rows)

def heatmap(df, title, fname, row="condition", order=None):
    ps = per_seed(df, ["side", row, "speed"])
    sides = ["right", "left"]
    fig, axes = plt.subplots(1, 2, figsize=(10, 0.45 * (len(order or ps[row].unique())) + 1.6), sharey=True)
    for ax, side in zip(axes, sides):
        m = ps[ps.side == side].groupby([row, "speed"]).p_fall.mean().unstack()
        if order:
            m = m.reindex(order)
        im = ax.imshow(m.values * 100, vmin=0, vmax=100, cmap="Reds", aspect="auto")
        ax.set_xticks(range(m.shape[1])); ax.set_xticklabels([f"{s:.1f}" for s in m.columns])
        ax.set_yticks(range(m.shape[0])); ax.set_yticklabels(m.index)
        ax.set_xlabel("Commanded speed (m/s)"); ax.set_title(f"{side.capitalize()} side")
        for i in range(m.shape[0]):
            for j in range(m.shape[1]):
                v = m.values[i, j] * 100
                ax.text(j, i, f"{v:.0f}", ha="center", va="center", fontsize=8,
                        color="white" if v > 60 else "black")
    fig.colorbar(im, ax=axes, label="Fall probability (%)", shrink=0.8)
    fig.suptitle(title)
    fig.savefig(OUT_DIR / fname, dpi=300, bbox_inches="tight")
    plt.show()

def kinematics_table(df, keys=("side", "condition", "speed")):
    cols = [c for c in df.columns if c.startswith(("rom_", "tauabs_", "sat_", "foot_", "trunk_", "base_vx", "vx_", "actstd_", "taudir_"))]
    return df.groupby(list(keys))[cols].mean().reset_index()

print("定義しました。")

In [ ]:
# --- 実験AB の集計 ---
df_AB = pd.read_csv(OUT_DIR / "expAB_symmetric_priority.csv")
tab_AB = fall_table(df_AB); tab_AB.to_csv(OUT_DIR / "table_AB_fall.csv", index=False)
tst_AB = treatment_tests(df_AB); tst_AB.to_csv(OUT_DIR / "table_AB_tests.csv", index=False)
kin_AB = kinematics_table(df_AB); kin_AB.to_csv(OUT_DIR / "table_AB_kinematics.csv", index=False)
print(tab_AB.pivot_table(index=["side", "condition"], columns="speed", values="mean").reindex(
    COND_ORDER, level=1).round(3).to_string())
heatmap(df_AB, "Fall probability: symmetric damping x10,000 at hip, knee, ankle", "fig_heatmap_AB.png",
        order=COND_ORDER)

In [ ]:
# --- 実験D1・D2 の集計(実行済みの場合) ---
p = OUT_DIR / "expD1_directional_dose.csv"
if p.exists():
    df_D1 = pd.read_csv(p)
    df_D1["cond"] = df_D1["joint_set"] + "_kd" + df_D1["kd"].astype(str)
    order = [f"{s}_kd{k}" for s in D1_SETS for k in D1_KDS]
    fall_table(df_D1, row="cond").to_csv(OUT_DIR / "table_D1_fall.csv", index=False)
    kinematics_table(df_D1, ("side", "joint_set", "kd", "speed")).to_csv(OUT_DIR / "table_D1_kinematics.csv", index=False)
    heatmap(df_D1, "Fall probability: direction-selective, velocity-dependent resistance", "fig_heatmap_D1.png",
            row="cond", order=order)
p = OUT_DIR / "expD2_directional_priority.csv"
if p.exists():
    df_D2 = pd.read_csv(p)
    fall_table(df_D2).to_csv(OUT_DIR / "table_D2_fall.csv", index=False)
    treatment_tests(df_D2).to_csv(OUT_DIR / "table_D2_tests.csv", index=False)
    kinematics_table(df_D2).to_csv(OUT_DIR / "table_D2_kinematics.csv", index=False)
    heatmap(df_D2, "Fall probability: direction-selective model, treatment conditions", "fig_heatmap_D2.png",
            order=COND_ORDER)

In [ ]:
# --- 実験C の集計(実行済みの場合)【修正版】---
# job_key(例: right_knee_x5000_s1)から side・target・damping_scale・seed を復元し、
# 復元できない行・数値が壊れている行は除外して、件数を報告する
p = OUT_DIR / "expC_single_joint_dose.csv"
if p.exists():
    raw_C = pd.read_csv(p, low_memory=False)
    m = raw_C["job_key"].astype(str).str.extract(r"^(right|left)_(.+)_x(\d+)_s(\d+)$")
    bad = m.isna().any(axis=1) | ~raw_C["fell"].astype(str).isin(["True", "False"])
    print(f"実験C: 全{len(raw_C)}行、うち壊れた行 {int(bad.sum())}行")
    if bad.any():
        print("  壊れた行の job_key(先頭10件):", raw_C.loc[bad, "job_key"].astype(str).unique()[:10].tolist())
        print("  元の damping_scale 欠損行数:", int(raw_C["damping_scale"].isna().sum()))
    df_C = raw_C.loc[~bad].copy()
    df_C["side"] = m.loc[~bad, 0]
    df_C["target"] = m.loc[~bad, 1]
    df_C["damping_scale"] = m.loc[~bad, 2].astype(int)
    df_C["seed"] = m.loc[~bad, 3].astype(int)
    df_C["fell"] = df_C["fell"].astype(str) == "True"
    df_C["speed"] = pd.to_numeric(df_C["speed"], errors="coerce")
    df_C = df_C.dropna(subset=["speed"])
    # 各ジョブが40行(4速度×10試行)そろっているか確認
    cnt = df_C.groupby("job_key").size()
    expected = {f"{s}_{t}_x{sc}_s{sd}" for s in ["right", "left"] for t in C_TARGETS for sc in C_SCALES for sd in SEEDS}
    missing = sorted(expected - set(cnt.index))
    print(f"  有効なジョブ {len(cnt)}/{len(expected)}、40行未満のジョブ {int((cnt < 40).sum())}、欠けているジョブ {len(missing)}")
    if missing:
        print("  欠けているジョブ(先頭10件):", missing[:10])
        print("  → §7のセルを再実行すると欠けている分だけ実行されます(その前に下のクリーニングを実行)")

    df_C["cond"] = df_C["target"] + "_x" + df_C["damping_scale"].astype(str)
    fall_table(df_C, row="cond").to_csv(OUT_DIR / "table_C_fall.csv", index=False)
    ps = per_seed(df_C, ["side", "target", "damping_scale", "speed"])
    fig, axes = plt.subplots(1, len(C_TARGETS), figsize=(4 * len(C_TARGETS), 3.2), sharey=True)
    for ax, tgt in zip(axes, C_TARGETS):
        for side, ls in [("right", "-"), ("left", "--")]:
            g = ps[(ps.target == tgt) & (ps.side == side) & (ps.speed == 0.5)].groupby("damping_scale").p_fall
            ax.errorbar(g.mean().index, g.mean() * 100, yerr=g.std(ddof=1) * 100, ls=ls, marker="o",
                        capsize=3, label=side)
        ax.set_xscale("log"); ax.set_title(tgt); ax.set_xlabel("Damping scale")
    axes[0].set_ylabel("Fall probability at 0.5 m/s (%)"); axes[0].legend()
    fig.savefig(OUT_DIR / "fig_dose_C.png", dpi=300, bbox_inches="tight"); plt.show()

## 9. 図2の改訂版(R2-#12)

右足の高さの時系列です。横軸は時間 [s]、縦軸は高さ [m]、主ピークと副次ピーク(閾値0.05 m)に印を付けます。
seed 1、0.5 m/s、初期状態のランダム化なしで、正常と右足関節 damping×1000 を比べます。

In [ ]:
def figure2(seed=1, scale=1000.0, speed=0.5, thresh=0.05):
    series = {}
    for label, jd in [("Unperturbed (damping scale 1)", None),
                      (f"Right ankle damping scale {scale:,.0f}", {"right_ankle_pitch_joint": scale})]:
        ts = run_batch(seed, [speed], joint_damping=jd, init_noise=0.0)
        v = ts["valid"][0]
        series[label] = ts["foot_z"][0, v, 1]
    fig, axes = plt.subplots(2, 1, figsize=(7.2, 4.6), sharex=True, sharey=True)
    for ax, (label, z) in zip(axes, series.items()):
        t = np.arange(len(z)) * STEP_DT
        ax.plot(t, z, color="#2b6cb0", lw=1.2)
        p = _peaks(z)
        main, small = p[z[p] >= thresh], p[z[p] < thresh]
        ax.plot(t[main], z[main], "o", ms=5, color="#2f855a", label=f"Main swing peaks (n={len(main)})")
        ax.plot(t[small], z[small], "x", ms=6, mew=1.5, color="#c53030", label=f"Secondary peaks (n={len(small)})")
        ax.axhline(thresh, color="gray", ls=":", lw=1, label=f"Classification threshold ({thresh} m)")
        ax.set_title(label, fontsize=10); ax.set_ylabel("Right foot height (m)")
        ax.legend(fontsize=7, loc="upper right", frameon=False)
    axes[1].set_xlabel("Time (s)")
    fig.tight_layout()
    for ext in ("png", "tiff"):
        fig.savefig(OUT_DIR / f"figure2_revised.{ext}", dpi=300)
    plt.show()
    return series

fig2_series = figure2()

## 10. 結果の書き出し

集計表・図・生CSVを1つのzipにまとめます。**このzipをそのまま共有してください**(時系列のnpzは大きいので含めていません)。

In [ ]:
import shutil, zipfile
zpath = Path("/content/q4_revision_results.zip")
with zipfile.ZipFile(zpath, "w", zipfile.ZIP_DEFLATED) as z:
    for f in sorted(OUT_DIR.glob("*")):
        if f.is_file():
            z.write(f, f.name)
shutil.copy(zpath, OUT_DIR / zpath.name)
print("作成しました:", zpath, f"({zpath.stat().st_size/1e6:.1f} MB)")
from google.colab import files
files.download(str(zpath))